<a href="https://www.kaggle.com/code/adityak1729/vit-gp?scriptVersionId=266159361" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')  # Columns: e.g., 'filename', 'label'
test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')    # Columns: e.g., 'filename', 'label'

classes = train_df['Label'].unique()
train_subset_df = pd.DataFrame()      

for label in classes:
    class_df = train_df[train_df['Label'] == label]
    sampled = class_df.sample(n=20, random_state=42) 
    train_subset_df = pd.concat([train_subset_df, sampled])

image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'  
train_subset_df['image_path'] = image_dir + train_subset_df['Filename']
test_df['image_path'] = image_dir + test_df['Filename']

print(f"Training subset shape: {train_subset_df.shape}")
print(f"Test set shape: {test_df.shape}")

Training subset shape: (200, 5)
Test set shape: (2700, 5)


In [8]:
import torch
print(torch.cuda.is_available())  # Should print True
print(torch.cuda.get_device_name(0))  # Prints GPU name, e.g., "NVIDIA GeForce RTX 3080"

True
Tesla P100-PCIE-16GB


In [15]:
from transformers import ViTImageProcessor, ViTForImageClassification, ViTConfig, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from PIL import Image
import torch

# Verify GPU
print(f"GPU Available: {torch.cuda.is_available()}")

# Load processor
model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)

# Raw ViT (from scratch, no pretrained weights) - Use this if you want raw; otherwise, comment out and use the fine-tuned line below
config = ViTConfig.from_pretrained(model_name, num_labels=len(classes))
model = ViTForImageClassification(config).to('cuda')

# OR Fine-tuned pretrained ViT (recommended for low-data; uncomment if preferred)
# model = ViTForImageClassification.from_pretrained(model_name, num_labels=len(classes), ignore_mismatched_sizes=True).to('cuda')

# Create Hugging Face Datasets from DFs
def create_hf_dataset(df):
    # Change 'Label' to your actual column name (run print(df.columns) to check)
    return Dataset.from_dict({"image": df['image_path'].tolist(), "label": df['Label'].tolist()})

train_dataset = create_hf_dataset(train_subset_df)
test_dataset = create_hf_dataset(test_df)

# Preprocess function (loads images on-the-fly, move to GPU)
def preprocess_function(examples):
    images = [Image.open(img).convert("RGB") for img in examples["image"]]
    processed = processor(images=images, return_tensors="pt")
    return {k: v.to('cuda') for k, v in processed.items()}

train_dataset = train_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)
print("Hello This reached here!!")

# Training arguments (defaults, no hyperparameter tuning) - Added TensorBoard support
training_args = TrainingArguments(
    output_dir="./vit_finetune",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",  # Use 'eval_strategy' for newer transformers versions; if error, try 'evaluation_strategy'
    save_strategy="epoch",
    learning_rate=2e-5,
    report_to="tensorboard",  # Enable TensorBoard logging
    logging_dir="./vit_finetune/runs",  # Directory for TensorBoard logs
    logging_steps=10,  # Log training loss every 10 steps for more console updates
)

# Compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Launch TensorBoard dashboard (run this before trainer.train() to see real-time updates)
%load_ext tensorboard
%tensorboard --logdir ./vit_finetune/runs

# Train and evaluate on full test set
trainer.train()
results = trainer.evaluate()
train_results = trainer.evaluate(train_dataset)  # Check training accuracy to confirm overfitting
print(f"Fine-Tuned ViT Training Accuracy: {train_results['eval_accuracy']:.4f}")
print(f"Fine-Tuned ViT Accuracy on Full Test Set: {results['eval_accuracy']:.4f}")

GPU Available: True


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Hello This reached here!!


<IPython.core.display.Javascript object>

Epoch,Training Loss,Validation Loss,Accuracy
1,2.561500,1.960903,0.228889
2,1.794500,1.754703,0.341481
3,1.624200,1.650815,0.389630


Fine-Tuned ViT Training Accuracy: 0.4250
Fine-Tuned ViT Accuracy on Full Test Set: 0.3896


In [16]:
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score

In [17]:
trainer.save_model("./raw_vit")

In [19]:
model_pre = ViTForImageClassification.from_pretrained(model_name, num_labels=len(classes), ignore_mismatched_sizes=True).to('cuda')

trainer_pre = Trainer(
    model=model_pre,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

%load_ext tensorboard
%tensorboard --logdir ./vit_finetune/runs


trainer_pre.train()
results_pre = trainer_pre.evaluate()
train_results_pre = trainer_pre.evaluate(train_dataset)
print(f"Pretrained ViT Training Accuracy: {train_results_pre['eval_accuracy']:.4f}")
print(f"Pretrained ViT Accuracy on Full Test Set: {results_pre['eval_accuracy']:.4f}")

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 124), started 0:28:39 ago. (Use '!kill 124' to kill it.)

<IPython.core.display.Javascript object>

Epoch,Training Loss,Validation Loss,Accuracy
1,2.160000,1.779383,0.485556
2,1.140900,1.420452,0.665556
3,0.964900,1.308756,0.715926


Pretrained ViT Training Accuracy: 0.9700
Pretrained ViT Accuracy on Full Test Set: 0.7159


In [20]:
trainer_pre.save_model("./pretrained_vit")

In [51]:
from torch.utils.data import Dataset
from PIL import Image

class SimpleImageDataset(Dataset):
    def __init__(self, df, image_processor):
        self.image_paths = df['image_path'].tolist()
        # Ensure your DataFrame's label column is also named 'Label'
        self.labels = df['Label'].tolist()
        self.processor = image_processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Return the raw PIL image and label
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        return {"image": image, "label": label}

In [52]:
def create_collate_fn(processor):
    def collate_fn(batch):
        # 'batch' is a list of dictionaries: [{'image': img1, 'label': 1}, {'image': img2, 'label': 5}, ...]
        
        # Extract the images and labels from the list of dictionaries
        images = [item['image'] for item in batch]
        labels = [item['label'] for item in batch]
        
        # Use the processor to create the 4D tensor batch. It handles everything.
        processed_batch = processor(images=images, return_tensors="pt")
        
        # Add the labels to the batch
        processed_batch['label'] = torch.tensor(labels)
        
        return processed_batch
    return collate_fn

In [53]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor

# --- SETUP (Do this once) ---
model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)

# 1. Create the datasets
# Use your actual dataframes here
train_ds_manual = SimpleImageDataset(train_subset_df, processor)
test_ds_manual = SimpleImageDataset(test_df, processor)

# 2. Create the collate function
my_collate_fn = create_collate_fn(processor)

# 3. Create the DataLoaders with the collate function
# This DataLoader now produces perfectly formed batches
train_dataloader_manual = DataLoader(train_ds_manual, batch_size=8, collate_fn=my_collate_fn)
test_dataloader_manual = DataLoader(test_ds_manual, batch_size=8, collate_fn=my_collate_fn)


# --- REVISED get_embeddings FUNCTION ---
def get_embeddings(model, dataloader, has_labels=True):
    model.eval()
    embeddings = []
    labels_list = [] if has_labels else None

    with torch.no_grad():
        for batch in dataloader:
            # The batch is already a perfect dictionary of tensors!
            # Just move the data to the GPU.
            inputs = {
                'pixel_values': batch['pixel_values'].to('cuda')
            }
            
            outputs = model(**inputs)
            emb = outputs.logits
            embeddings.append(emb.cpu().numpy())
            
            if has_labels:
                labels_list.append(batch['label'].numpy())

    embeddings = np.vstack(embeddings)
    if has_labels:
        labels_list = np.hstack(labels_list)
        return embeddings, labels_list
    
    return embeddings

# --- HOW TO USE IT ---
# model.classifier = nn.Identity() # Make sure to remove the classification head
# train_emb, train_labels = get_embeddings(model, train_dataloader_manual)
# test_emb, test_labels = get_embeddings(model, test_dataloader_manual)

In [55]:
original_classifier_raw = model.classifier
model.classifier = nn.Identity() # Make sure to remove the classification head
train_emb, train_labels = get_embeddings(model, train_dataloader_manual)
test_emb, test_labels = get_embeddings(model, test_dataloader_manual)

In [56]:
model.classifier = original_classifier_raw

In [68]:
original_classifier_pre = model_pre.classifier
model_pre.classifier = nn.Identity()
train_emb_pre, train_labels_pre = get_embeddings(model_pre, train_dataloader_manual)
test_emb_pre, test_labels_pre = get_embeddings(model_pre, test_dataloader_manual)
model_pre.classifier = original_classifier_pre

In [64]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

# --- 2. Create a kernel with tunable hyperparameters ---
# Let the GP find the best length_scale instead of fixing it to 1.0
# The bounds (1e-3, 1e3) give it a wide range to search.
tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3))

In [69]:
# kernel = 1.0 * RBF(1.0)
gp_raw = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)
gp_raw.fit(train_emb, train_labels)
pred_gp_raw = gp_raw.predict(test_emb)
acc_gp_raw = accuracy_score(test_labels, pred_gp_raw)
print(f"GP on Raw ViT Embeddings - Test Accuracy: {acc_gp_raw:.4f}")

/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/_gpc.py:477: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


GP on Raw ViT Embeddings - Test Accuracy: 0.5411


In [70]:
gp_pre = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)
gp_pre.fit(train_emb_pre, train_labels_pre)
pred_gp_pre = gp_pre.predict(test_emb_pre)
acc_gp_pre = accuracy_score(test_labels_pre, pred_gp_pre)
print(f"GP on Pretrained ViT Embeddings - Test Accuracy: {acc_gp_pre:.4f}")

GP on Pretrained ViT Embeddings - Test Accuracy: 0.1115


In [77]:
from sklearn.linear_model import LogisticRegression

# 1. Scale the data (still good practice)
scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

# 2. Train a simple, powerful linear model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(train_emb_pre_scaled, train_labels)

# 3. Evaluate
accuracy_log_reg_pretrained = log_reg.score(test_emb_pre_scaled, test_labels)
print(f"Logistic Regression on Pretrained Embeddings - Test Accuracy: {accuracy_log_reg_pretrained:.4f}")

Logistic Regression on Pretrained Embeddings - Test Accuracy: 0.8478


In [78]:
from sklearn.linear_model import LogisticRegression

# 1. Scale the data (still good practice)
scaler = StandardScaler()
train_emb_raw_scaled = scaler.fit_transform(train_emb)
test_emb_raw_scaled = scaler.transform(test_emb)

# 2. Train a simple, powerful linear model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(train_emb_raw_scaled, train_labels)

# 3. Evaluate
accuracy_log_reg_raw = log_reg.score(test_emb_raw_scaled, test_labels)
print(f"Logistic Regression on Raw Embeddings - Test Accuracy: {accuracy_log_reg_raw:.4f}")

Logistic Regression on Raw Embeddings - Test Accuracy: 0.5830


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [76]:
# using PCA + GP on Pretrained VIT Embeddings
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score

scaler = StandardScaler()
train_emb_scaled = scaler.fit_transform(train_emb_pre)
test_emb_scaled = scaler.transform(test_emb_pre)

pca = PCA(n_components=64, random_state=42)
train_emb_pca = pca.fit_transform(train_emb_scaled)
test_emb_pca = pca.transform(test_emb_scaled)

print(f"Original embedding shape: {train_emb_scaled.shape}")
print(f"Shape after PCA: {train_emb_pca.shape}")

tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3))
gp_pca = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)

print("\nTraining GP on PCA-reduced embeddings...")
gp_pca.fit(train_emb_pca, train_labels)

pred_gp_pca = gp_pca.predict(test_emb_pca)
acc_gp_pca = accuracy_score(test_labels, pred_gp_pca)

print(f"\nGP on PCA Pretrained Embeddings - Test Accuracy: {acc_gp_pca:.4f}")

Original embedding shape: (200, 768)
Shape after PCA: (200, 64)

Training GP on PCA-reduced embeddings...


/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/_gpc.py:477: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)



GP on PCA Pretrained Embeddings - Test Accuracy: 0.8470


In [81]:
#GP with linear kernel on Pretrained embeddings

from sklearn.gaussian_process.kernels import DotProduct

# On the original, high-dimensional scaled data
# A DotProduct kernel is scikit-learn's linear kernel for GPs
linear_kernel = DotProduct(sigma_0=1.0, sigma_0_bounds="fixed")

gp_linear = GaussianProcessClassifier(kernel=linear_kernel, random_state=42, n_jobs=-1)
gp_linear.fit(train_emb_scaled, train_labels)

acc_gp_linear = gp_linear.score(test_emb_scaled, test_labels)
print(f"\nGP with Linear Kernel on Pretrained Embeddings - Test Accuracy: {acc_gp_linear:.4f}")


GP with Linear Kernel on Pretrained Embeddings - Test Accuracy: 0.8356


In [84]:
# using PCA + GP on Pretrained Raw Embeddings
scaler = StandardScaler()
train_emb_scaled_raw = scaler.fit_transform(train_emb)
test_emb_scaled_raw = scaler.transform(test_emb)

pca = PCA(n_components=64, random_state=42)
train_emb_pca_raw = pca.fit_transform(train_emb_scaled_raw)
test_emb_pca_raw = pca.transform(test_emb_scaled_raw)

print(f"Original embedding shape: {train_emb_scaled_raw.shape}")
print(f"Shape after PCA: {train_emb_pca_raw.shape}")

tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3))
gp_pca_raw = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)

print("\nTraining GP on PCA-reduced embeddings...")
gp_pca.fit(train_emb_pca_raw, train_labels)

pred_gp_pca_raw = gp_pca.predict(test_emb_pca_raw)
acc_gp_pca_raw = accuracy_score(test_labels, pred_gp_pca_raw)

print(f"\nGP on PCA Raw Embeddings - Test Accuracy: {acc_gp_pca_raw:.4f}")

Original embedding shape: (200, 768)
Shape after PCA: (200, 64)

Training GP on PCA-reduced embeddings...


/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/_gpc.py:477: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)



GP on PCA Raw Embeddings - Test Accuracy: 0.5493


In [87]:
#GP with linear kernel on Raw embeddings

from sklearn.gaussian_process.kernels import DotProduct

# On the original, high-dimensional scaled data
# A DotProduct kernel is scikit-learn's linear kernel for GPs
linear_kernel = DotProduct(sigma_0=1.0, sigma_0_bounds="fixed")

gp_linear = GaussianProcessClassifier(kernel=linear_kernel, random_state=42, n_jobs=-1)
gp_linear.fit(train_emb_scaled_raw, train_labels)

acc_gp_linear_raw = gp_linear.score(test_emb_scaled_raw, test_labels)
print(f"\nGP with Linear Kernel on Pretrained Embeddings - Test Accuracy: {acc_gp_linear_raw:.4f}")


GP with Linear Kernel on Pretrained Embeddings - Test Accuracy: 0.5844


In [88]:
print("\nSummary:")
print(f"Raw ViT (direct classification) Test Accuracy: 0.3896")
print(f"Pretrained ViT (direct classification) Test Accuracy: {results_pre['eval_accuracy']:.4f}")
print(f"GP on Raw ViT Embeddings Test Accuracy (using tuned_kernel): {acc_gp_raw:.4f}")
print(f"GP on Pretrained ViT Embeddings Test Accuracy (using tuned_kernel): {acc_gp_pre:.4f}")
print(f"Logistic Regression on Raw Embeddings - Test Accuracy: {accuracy_log_reg_raw:.4f}")
print(f"Logistic Regression on Pretrained Embeddings - Test Accuracy: {accuracy_log_reg_pretrained:.4f}")
print(f"GP on PCA Pretrained Embeddings - Test Accuracy: {acc_gp_pca:.4f}")
print(f"GP with Linear Kernel on Pretrained Embeddings - Test Accuracy: {acc_gp_linear:.4f}")
print(f"GP on PCA Raw Embeddings - Test Accuracy: {acc_gp_pca_raw:.4f}")
print(f"GP with Linear Kernel on Raw Embeddings - Test Accuracy: {acc_gp_linear_raw:.4f}")


Summary:
Raw ViT (direct classification) Test Accuracy: 0.3896
Pretrained ViT (direct classification) Test Accuracy: 0.7159
GP on Raw ViT Embeddings Test Accuracy (using tuned_kernel): 0.5411
GP on Pretrained ViT Embeddings Test Accuracy (using tuned_kernel): 0.1115
Logistic Regression on Raw Embeddings - Test Accuracy: 0.5830
Logistic Regression on Pretrained Embeddings - Test Accuracy: 0.8478
GP on PCA Pretrained Embeddings - Test Accuracy: 0.8470
GP with Linear Kernel on Pretrained Embeddings - Test Accuracy: 0.8356
GP on PCA Raw Embeddings - Test Accuracy: 0.5493
GP with Linear Kernel on Raw Embeddings - Test Accuracy: 0.5844
